# LizyML Widget Tutorial — Regression

This tutorial demonstrates using LizyML Widget for a **regression** task
with the [California Housing](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset) dataset.

You will learn:
1. Loading real-world data into the widget
2. Reviewing auto-detected settings
3. Configuring and running Fit
4. Reviewing regression-specific metrics and plots
5. Hyperparameter tuning
6. Running inference on new data

## 0. Installation

```bash
pip install lizyml-widget lizyml[plots,tuning,calibration,explain]
```

## 1. Load the California Housing Dataset

This dataset contains 20,640 samples with 8 features describing California
housing block groups. The target is the median house value (in $100,000s).

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
df = data.frame
print(f"Shape: {df.shape}")
print(f"Target: MedHouseVal (median house value in $100k)")
df.head()

## 2. Launch the Widget

Create a widget instance, load the data, and display it.

In [ ]:
from lizyml_widget import LizyWidget

w = LizyWidget()
w.load(df, target="MedHouseVal")
w

## 3. Data Tab — Auto-Detection

The widget auto-detects:
- **Task**: `regression` (continuous target values)
- **CV**: `kfold` (default for regression)
- **Columns**: All 8 features included

In [ ]:
print(f"Task:     {w.task}")
print(f"CV:       {w.cv_method} ({w.cv_n_splits} folds)")
print(f"Shape:    {w.df_shape}")

## 4. Config — Model Configuration

Set model hyperparameters. For regression, we use typical LightGBM settings.

In [ ]:
w.set_config({
    "model": {
        "name": "lgbm",
        "params": {
            "n_estimators": 500,
            "learning_rate": 0.05,
            "max_depth": 6,
        },
    },
    "training": {
        "seed": 42,
        "early_stopping": {"enabled": True, "rounds": 50},
    },
    "evaluation": {
        "metrics": ["rmse", "mae", "r2"],
    },
})

## 5. Run Fit

Train the model with cross-validation. Progress is displayed in the widget.

In [ ]:
w.fit()

## 6. Results — Regression Metrics & Plots

After fitting, view metrics in the Results tab. Key regression metrics:
- **RMSE** — Root Mean Squared Error
- **MAE** — Mean Absolute Error
- **R²** — Coefficient of Determination

Regression-specific plots:
- **Residuals** — Predicted vs actual residuals
- **OOF Distribution** — Out-of-fold prediction distribution
- **Feature Importance** — Split, Gain, and SHAP variants

In [ ]:
summary = w.get_fit_summary()
if summary:
    print(f"Fold count: {summary.fold_count}")
    print(f"Metrics:    {summary.metrics}")

## 7. Tune — Hyperparameter Optimization

Use Optuna to search for optimal hyperparameters.

In [ ]:
w.tune()

tune_summary = w.get_tune_summary()
if tune_summary:
    print(f"Best score:  {tune_summary.best_score:.4f}")
    print(f"Metric:      {tune_summary.metric_name} ({tune_summary.direction})")
    print(f"Best params: {tune_summary.best_params}")
    print(f"Trials:      {len(tune_summary.trials)}")

## 8. Inference — Predict on New Data

Use the trained model to predict housing values for new samples.

In [ ]:
# Use a subset of the original data as "new" data for demonstration
df_test = df.drop(columns=["MedHouseVal"]).tail(10)

result = w.predict(df_test)
if result:
    print(f"Predictions shape: {result.predictions.shape}")
    result.predictions.head()

## 9. Save & Load Config

Save the current configuration for reproducibility.

In [ ]:
# Save config
w.save_config("regression_config.yaml")

# To load in another session:
# w2 = LizyWidget()
# w2.load(df, target="MedHouseVal")
# w2.load_config("regression_config.yaml")

## 10. Export Code

In [ ]:
# Export standalone training code (no LizyML dependency needed)
export_path = w.export_code("./exported_code")
print(f"Code exported to: {export_path}")
print("Generated files:")
for f in sorted(export_path.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(export_path)}")